# TPCのゲインマップ：rawツリー → 1-hitスペクトル → ピークフィット → Ka/Kb

`RawTreeWriter` のFADC波形から、アノードch × カソードchの領域ごとに
**E [keV] = Ka × Q [ADU × sample] + Kb [keV]** を決めます。
SMILE-3の `CalcFADC` は電荷をpCに換算しないので、Kaの単位は **keV/(ADU × sample)** です。
較正係数は、この電荷の単位に合わせて求めます。

FADC ch = 4 × board ID + ch（0〜47）= strip / 32。panel = board ID / 3 = FADC ch / 12。
同じパネルの12 × 12組を4パネル、計576組が対象です。
線源の同定とフィット範囲はユーザーが指定します。未知の線源や不明なピークを推測で較正しません。

In [ ]:
# ===== 入力と出力(絶対パス。データ量が大きいので、リポジトリの外の NAS などに置く)=====
from pathlib import Path
# 入力: TPC_RawTreeWriter/raw を含むROOTファイル(同じ測定条件のもの)
RAWS = [
    Path("/mnt/nas_eureka/onoda/<書き換え>/tpc_raw_20261111_run71.root"),
]
# 出力: このノートブックが書くファイルの置き場
OUT = Path("/mnt/nas_eureka/onoda/<書き換え>/tpc_gain_20261111_run71")

for p in [*RAWS, OUT]:
    if "<" in str(p):
        raise ValueError(f"パスを書き換えてください: {p}")
OUT.mkdir(parents=True, exist_ok=True)
print("入力:"); [print("  ", p) for p in RAWS]
print("出力:", OUT)


In [ ]:
import sys, json, subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from IPython.display import Image, display
import ROOT
ROOT.gROOT.SetBatch(True)
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'run_script/run_chain.py').exists())
sys.path.insert(0, str(REPO / "scripts/analysis"))
from tpc_raw_io import raw_tree_path, run_info_path, storage_threshold
SCRIPTS = REPO / 'scripts/calib'
sys.path.insert(0, str(SCRIPTS))
PYTHON = '/usr/bin/python3'  # numpy / PyROOT のある Python
TREE = 'TPC_RawTreeWriter/raw'

## 1. 入力ROOTファイル

このノートブックは **既存の `TPC_RawTreeWriter/raw` を読む**もので、生データを自動で再解析しません。
冒頭の `RAWS` を実際のROOTファイルに変更します。複数ファイルは同じ測定条件のものを指定してください。
run70 は ETCC、run71〜73 は TPC_cal です。測定条件と線源を確認し、モードだけで較正に適しているとは判断しません。
前の積算処理が作った `root_data/20261111_tracks/run*.root` には raw ツリーが無いため、入力にはできません。

入力・出力はデータ量が大きいのでリポジトリの外(NASなど)に置き、冒頭の `RAWS` と `OUT` に絶対パスで書きます。
ROOTファイルがまだ無い場合は、ターミナルで例えば次を実行します。
`python3` は EricaPy をビルドした Python、ノートブックは numpy / PyROOT のある「EricaNext」カーネルを使います。

```bash
source ~/opt/setup_ericanext.sh
cd ~/ANLNext/Erica_git_dev/EricaNext
python3 run_script/run_chain.py smile3_tpc_raw --compression 505 \
  -o /mnt/nas_eureka/onoda/<書き換え>/tpc_raw_20261111_run71.root \
  --data-dir ~/QL_PC/storage_g0/20261111_run71 \
  --extra-data-dir ~/QL_PC/storage_g1/20261111_run71
```

`smile3_tpc_raw` は全ヒット・レコード・全チャンネルの波形を保存します。
波形を閾値で間引いたファイルでは小さい信号が欠けるので、ゲイン較正には間引かないものを使います。

In [ ]:
for path in RAWS:
    if not path.is_file(): raise FileNotFoundError(path)
    f = ROOT.TFile.Open(str(path))
    if not f or f.IsZombie(): raise RuntimeError(f'ROOTファイルを開けません: {path}')
    t = f.Get(raw_tree_path(path, TREE))
    if not t: raise RuntimeError(f'{path}: {TREE} がありません')
    required = ['fadc_ch', 'fadc_wave']
    missing = [b for b in required if not t.GetBranch(b)]
    if missing: raise RuntimeError(f'{path}: 必要なブランチがありません: {missing}')
    print(path.name, t.GetEntries(), 'events')
    f.Close()

## 2. 電荷の計算と1-hitスペクトル

`scripts/calib/make_tpc_spec1hit.py` で `CalcFADC` と同じ積分を行います。
ベースラインは最初の30 sampleの平均、積分範囲はその後から波形末尾まで。
ベースラインから20 ADUを超えるsampleだけを足します（1 sample = 20 ns）。
カソード波形はrawツリーですでに反転済みです。

選別は「全μPICを通じてアノードで電荷のあるchが1本、カソードでも1本だけ、かつ両者が同じμPIC」です。
別μPICのカソードchにも閾値を超えるサンプルがあれば除外します。ボード単位（4 chの和）と、隣り合う2 ch（FADC ch 2k と 2k+1）の和の1-hitも、同様に全体でアノード・カソード1組ずつとします。
旧条件（カソードは同じμPIC内だけで数える）で作ったスペクトル・係数は再生成してください。出力ROOTの `selection_metadata_json` に選別条件を記録します。
保存閾値は0を推奨します。異なる閾値・測定条件のスペクトルは混ぜません。
既存の `spec1hit.root` は、中の `selection_metadata_json` の入力ファイル・`BASE_WINDOW`・`THRESHOLD`・選別条件が
今の設定と一致し、入力ROOTより新しい場合だけ再利用します(作成を飛ばすので Run All でもすぐ次に進みます)。
どれかが違えば理由を表示して作り直します。強制的に作り直すときは `REMAKE_SPEC = True` にします。

In [ ]:
BASE_WINDOW = 30   # ベースラインを取る長さ [sample]
THRESHOLD = 20.0   # ベースラインからの積分閾値 [ADU]
REMAKE_SPEC = False  # Trueなら既存のspec1hit.rootがあっても作り直す
for path in RAWS:
    f = ROOT.TFile.Open(str(path))
    info = f.Get(run_info_path(path))
    if not info or not info.GetEntries(): raise RuntimeError(f'{path}: run_infoがありません')
    info.GetEntry(0)
    if storage_threshold(info) != 0: raise RuntimeError('ゲイン較正には全波形を保存したrawを用意してください')
    if not 0 < BASE_WINDOW < info.n_fadc_clk: raise ValueError('BASE_WINDOWが不正です')
    f.Close()
SPEC_FILE = OUT / 'spec1hit.root'

def spec_mismatch(spec):
    """既存のspec1hit.rootが今の設定で作られたものか調べ、違う点を返す(空なら再利用可)。"""
    f = ROOT.TFile.Open(str(spec))
    if not f or f.IsZombie(): return ['ROOTファイルを開けない']
    obj = f.Get('selection_metadata_json')
    meta = json.loads(str(obj.GetString())) if obj else None
    f.Close()
    if meta is None: return ['selection_metadata_jsonがない(旧条件のファイル)']
    why = []
    if meta.get('inputs') != [str(p.absolute()) for p in RAWS]: why.append('入力ファイルが違う')
    if meta.get('base_window_samples') != BASE_WINDOW: why.append('BASE_WINDOWが違う')
    if meta.get('threshold_ADU') != THRESHOLD: why.append('THRESHOLDが違う')
    if meta.get('selection') != 'one anode channel and one cathode channel globally; same panel':
        why.append('1-hitの選別条件が違う')
    if 'pair2_selection' not in meta: why.append('2 ch単位のスペクトルがない(古いファイル)')
    if any(p.stat().st_mtime > spec.stat().st_mtime for p in RAWS): why.append('入力がspec1hit.rootより新しい')
    return why

why = ['REMAKE_SPEC=True'] if REMAKE_SPEC else (spec_mismatch(SPEC_FILE) if SPEC_FILE.exists() else ['spec1hit.rootがない'])
if why:
    print('スペクトルを作成します:', ', '.join(why))
    cmd = [PYTHON, str(SCRIPTS / 'make_tpc_spec1hit.py'), '-o', str(SPEC_FILE), *map(str, RAWS),
           '--base-window', str(BASE_WINDOW), '--threshold', str(THRESHOLD), '--overwrite']
    print(' '.join(cmd)); subprocess.run(cmd, check=True)
else:
    print('既存のスペクトルを使います(入力・BASE_WINDOW・THRESHOLD・選別条件が一致):', SPEC_FILE)
if (OUT / 'qa/spec1hit_board.png').exists():
    display(Image(filename=str(OUT / 'qa/spec1hit_board.png')))

## 3. 組ごとの統計とスペクトル

1-hitスペクトルは2000ビン、0〜10000 ADU × sample（5 ADU × sample/bin）です。
ピークが軸外ならスペクトル作成スクリプトの範囲を見直し、端で切れた山をフィットしません。
まず統計マップと任意の組のスペクトルを見て、線源に対応する2本のピークを確認します。

In [ ]:
f = ROOT.TFile.Open(str(SPEC_FILE))
spectra = {}
for a in range(48):
    for c in range(12*(a//12), 12*(a//12+1)):
        h = f.Get(f'Anode_1Hit_A{a:02d}_C{c:02d}')
        if h:
            spectra[a,c] = h.Clone(f'gain_A{a}_C{c}'); spectra[a,c].SetDirectory(0)
f.Close()
counts = np.full((48,48), np.nan)
for (a,c), h in spectra.items(): counts[c,a] = h.Integral()
fig, ax = plt.subplots(figsize=(9,7))
m = ax.imshow(np.ma.masked_invalid(np.ma.masked_less(counts,1)), origin='lower',
              norm=LogNorm(vmin=1, vmax=max(2, np.nanmax(counts) if spectra else 2)))
fig.colorbar(m, ax=ax, label='1-hit events'); ax.set(xlabel='Anode FADC ch', ylabel='Cathode FADC ch')
fig.tight_layout(); fig.savefig(OUT / 'counts.png', dpi=150); plt.show()
print('スペクトルのある組:', len(spectra))

In [ ]:
PAIR = (12, 12)  # 表示したい組 (anode FADC ch, cathode FADC ch)。同じμPIC(ch // 12 が同じ)の組を書く
h = spectra.get(PAIR)
if h is None:
    print(f'A{PAIR[0]} C{PAIR[1]} のスペクトルはありません。ある組の例:', sorted(spectra)[:5])
else:
    x = np.array([h.GetBinCenter(i) for i in range(1,h.GetNbinsX()+1)])
    y = np.array([h.GetBinContent(i) for i in range(1,h.GetNbinsX()+1)])
    plt.figure(figsize=(10,4)); plt.step(x,y,where='mid')
    plt.xlabel('Anode charge [ADU x sample]'); plt.ylabel('Events / 5 ADU x sample')
    plt.title(f'A{PAIR[0]} C{PAIR[1]}'); plt.show()

### μPICごとの1-hitスペクトル一覧（FADC ch単位・2 ch単位・ボード単位）

各μPICの1-hitスペクトルを1枚に並べます。列がアノード、行がカソード（上ほど大きい番号）、各図右上の数字はイベント数です。

- FADC ch単位（12 × 12組）: `spec1hit_upic{0..3}.png`
- 2 ch単位（隣り合う2 chの和、6 × 6組、番号 G = FADC ch // 2、つまりch 2Gと2G+1）: `spec1hit_2ch_upic{0..3}.png`
- ボード単位（4 chの和、3 × 3組、ボード番号 B = FADC ch // 4）: `spec1hit_board_upic{0..3}.png`

横軸は同じ図の全組で共通で、パネル内の全スペクトルの99.5%点までに狭めています（`VIEW_XMAX` で固定可）。
縦軸は組ごとに自動調整なので、高さの比較ではなくピーク位置・形の比較に使います。灰色はスペクトルのない組です。


In [ ]:
VIEW_REBIN = 4        # 表示用にまとめる元ビン数 [bins]（4なら20 ADU x sample/bin）
VIEW_XMAX = None      # 横軸上限 [ADU x sample]。Noneならパネル内の99.5%点から自動

def plot_spectra_grid(spec, ids, prefix, title, png, figsize, fs=7, fit_results=None, ok=None):
    """spec[(anode, cathode)] のTH1を、列=アノード・行=カソード(上が大きい番号)で並べる。
    fit_results={(a,c): [line0, line1]} を渡すとフィット曲線と中心を重ね、ok(集合)に入る組をOK、他をNGと表示する。"""
    pairs = [(a, c) for c in ids for a in ids if (a, c) in spec]
    if not pairs: print(f'{title}: スペクトルなし'); return
    h0 = spec[pairs[0]]
    nb = h0.GetNbinsX() // VIEW_REBIN * VIEW_REBIN
    edges = np.array([h0.GetBinLowEdge(i) for i in range(1, nb+2)])[::VIEW_REBIN]
    ys = {p: np.array([spec[p].GetBinContent(i) for i in range(1, nb+1)]).reshape(-1, VIEW_REBIN).sum(1)
          for p in pairs}
    xmax = VIEW_XMAX
    if xmax is None:
        cum = np.cumsum(sum(ys.values())); cum = cum/cum[-1] if cum[-1] > 0 else cum
        xmax = edges[min(np.searchsorted(cum, 0.995)+2, len(edges)-1)]
    n = len(ids)
    fig, axs = plt.subplots(n, n, figsize=figsize, sharex=True, squeeze=False)
    for i, c in enumerate(reversed(ids)):
        for j, a in enumerate(ids):
            ax = axs[i, j]
            ax.tick_params(labelsize=fs-2, length=2, pad=1)
            if (a, c) not in ys:
                ax.set_facecolor('0.9'); ax.set_yticks([]); continue
            y = ys[a, c]
            ax.stairs(y, edges, lw=0.6, color='C0', fill=False)
            for k, r in enumerate((fit_results or {}).get((a, c), [])):
                xx = np.linspace(*r['window'], 100)
                # ROOT fit values are counts per FIT_REBIN bins; convert to VIEW_REBIN bins.
                ax.plot(xx, [r['fn'].Eval(v)*VIEW_REBIN/FIT_REBIN for v in xx], color=f'C{k+1}', lw=0.8)
                ax.axvline(r['mu'], ls=':', color='k', lw=0.6)
            if ok is not None:
                good = (a, c) in ok
                ax.text(0.03, 0.95, 'OK' if good else 'NG', transform=ax.transAxes, ha='left', va='top',
                        fontsize=fs-1, fontweight='bold', color='g' if good else 'r')
                if not good: ax.set_facecolor('#fde8e8')
            ax.set_ylim(0, max(1, y[edges[:-1] < xmax].max()*1.1))
            ax.text(0.97, 0.95, f'{int(y.sum())}', transform=ax.transAxes,
                    ha='right', va='top', fontsize=fs-2)
            ax.yaxis.set_major_locator(plt.MaxNLocator(2))
    for ax in axs.flat: ax.set_xlim(0, xmax); ax.xaxis.set_major_locator(plt.MaxNLocator(2))
    for j, a in enumerate(ids):
        axs[-1, j].set_xlabel(f'A{prefix}{a}', fontsize=fs); axs[0, j].set_title(f'A{prefix}{a}', fontsize=fs)
    for i, c in enumerate(reversed(ids)): axs[i, 0].set_ylabel(f'C{prefix}{c}', fontsize=fs)
    fig.suptitle(f'{title}\n(x: Anode charge [ADU x sample], 0–{xmax:g}; '
                 f'y: events / {5*VIEW_REBIN} ADU x sample; number = events)', fontsize=fs+3)
    fig.tight_layout(rect=(0, 0, 1, 0.97), h_pad=0.2, w_pad=0.2)
    fig.savefig(png, dpi=120); plt.show(); plt.close(fig)

for panel in range(4):
    plot_spectra_grid(spectra, range(12*panel, 12*(panel+1)), '', f'μPIC{panel}: 1-hit spectra',
                      OUT/f'spec1hit_upic{panel}.png', figsize=(16, 20))

In [ ]:
# 2 ch単位・ボード単位の1-hit（全体でアノード1組・カソード1組、同じμPIC、組内のchの和）
def load_group_spectra(tag, size):
    """spec1hit.root の Anode_1Hit_<tag>_A<i>_C<j> を {(i, j): TH1} で返す(size = 1組のch数)。"""
    per_panel = 12 // size
    out = {}
    f = ROOT.TFile.Open(str(SPEC_FILE))
    for a in range(48 // size):
        for c in range(per_panel*(a//per_panel), per_panel*(a//per_panel+1)):
            h = f.Get(f'Anode_1Hit_{tag}_A{a}_C{c}')
            if h: out[a,c] = h.Clone(f'{tag}_A{a}_C{c}'); out[a,c].SetDirectory(0)
    f.Close()
    return out
pair2_spectra = load_group_spectra('2ch', 2)
board_spectra = load_group_spectra('board', 4)
print('スペクトルのある組: 2 ch単位', len(pair2_spectra), '/ ボード単位', len(board_spectra))
for panel in range(4):
    plot_spectra_grid(pair2_spectra, range(6*panel, 6*(panel+1)), 'G',
                      f'μPIC{panel}: 2-ch 1-hit spectra (FADC ch = 2 x G + 0..1)',
                      OUT/f'spec1hit_2ch_upic{panel}.png', figsize=(12, 15), fs=8)
    plot_spectra_grid(board_spectra, range(3*panel, 3*(panel+1)), 'B',
                      f'μPIC{panel}: board 1-hit spectra (FADC ch = 4 x board + 0..3)',
                      OUT/f'spec1hit_board_upic{panel}.png', figsize=(9, 11), fs=9)

## 4. 線のエネルギーとフィット範囲を設定

`ENERGIES_KEV` に **測定で確認した2本の線**を小さい順に指定します。
例えばBa/CuのX線と確認できた測定なら `[8.04, 30.85]` keVですが、今回のrunの線源を意味しません。
フィット範囲は `[(低エネルギー線のQ下限,Q上限), (高エネルギー線のQ下限,Q上限)]` の形で、次の順に決めます。

1. **ボード単位でピーク位置を決める**: ボード単位の1-hitスペクトル（3 × 3組/μPIC）を
   `BOARD_WINDOWS[(anode_board, cathode_board)]`（無ければ `DEFAULT_WINDOWS`）の範囲でフィットします。
   結果は `spec1hit_board_fit_upic{0..3}.png` で確認し、NGのボードは `BOARD_WINDOWS` を直して再実行します。
2. **各ch組の範囲をボードの結果から決める**: ch組 (a, c) の範囲は、それを含むボード (a//4, c//4) のフィットがOKなら
   各線で `ボードの中心 ± AUTO_WINDOW_NSIGMA × ボードのσ` にします。ボードのσは中の4 chのゲインのばらつきも含みます。
   2本の範囲が重なる場合は、2つの中心の中点で分けます。
   このときch組のフィットは、ボードの中心を初期値にします。`BOARD_SIGMA_RANGE = (下限, 上限)` を指定すると、
   ガウスのσを (下限, 上限) × ボードのσ の範囲に制限します（既定は `None` で制限なし）。
   ボードのσは4 chのゲインのばらつきを含むので、chのσはふつうそれ以下です。制限端に張り付いた組は「幅が制限端」でNGになります。
3. 優先順位は `WINDOWS[(a,c)]`（個別指定）> ボードから自動 > `DEFAULT_WINDOWS` です。
   ボードのフィットがNGなら、そのボードのch組は `DEFAULT_WINDOWS` を使います。

4. 2 ch単位（隣り合う2 chの和）も同じ方法でフィットし、`spec1hit_2ch_fit_upic{0..3}.png` に描きます。
   ch組より統計が多いので、ch組で決まらない場合の目安にします。この結果はゲインマップの係数には使いません。

どの範囲を使ったかは `gain_map.json` の `window_source` に記録します。
範囲が1つも決まらない組は較正しません。

背景込みのガウス（gaus + pol1）を指定範囲でフィットします。
統計、fit status、自由度、χ²/ndf、中心・幅の制限への張り付き、中心誤差、有意度を確認します。
有意度はガウスの高さ / その誤差で、背景と強く相関する場合は厳しい判定になります。
判定に通った2点を結ぶ直線でKa/Kbを求めます（原点は追加しません）。

### ピーク中心の誤差から係数の誤差へ

ピーク中心を μ1, μ2、その標準誤差を s1, s2、対応するエネルギーを E1, E2 とします。
Δ = μ2 − μ1、ΔE = E2 − E1 とすると、2点を結ぶ係数は
`Ka = ΔE / Δ`、`Kb = E1 − Ka × μ1` です。
2点なので中心値を通る直線は一意ですが、係数にも不確かさがあります。

2つのピーク中心の誤差を独立と仮定すると、一次の誤差伝播は次の式になります。

- `Var(Ka) = (ΔE / Δ²)² × (s1² + s2²)`
- `Var(Kb) = (ΔE / Δ²)² × (μ2² × s1² + μ1² × s2²)`
- `Cov(Ka, Kb) = −(ΔE / Δ²)² × (μ2 × s1² + μ1 × s2²)`

分散の平方根が `Ka_stat_error`・`Kb_stat_error` です。
ガウスの幅σではなく、ROOTの `GetParError(1)` が返す**中心位置の標準誤差**を使います。
各ピークfit内の他パラメータとの相関はその誤差に含まれますが、2つのfit間の相関は0と仮定します。
線のエネルギーは固定値とし、線の同定・背景モデル・fit範囲などの系統誤差は含めません。
誤差が大きく一次近似が適切でない場合は、サンプリングなどで別途検証が必要です。

係数の誤差は強く相関するため、固定した電荷Qで `E = Ka×Q + Kb` の係数由来の分散を求めるときは
`Q² Var(Ka) + Var(Kb) + 2Q Cov(Ka,Kb)` を使います。電荷Q自身の測定誤差は別に扱います。


In [ ]:
ENERGIES_KEV = [8.04, 30.85]  # 例: [8.04, 30.85]。線源を確認して設定 [keV]
DEFAULT_WINDOWS = None  # 例: [(Q1下限,Q1上限), (Q2下限,Q2上限)] [ADU x sample]
WINDOWS = {}  # {(anode_ch, cathode_ch): [(Q1下限,Q1上限), (Q2下限,Q2上限)]}
# ボード単位のフィット範囲 {(anode_board, cathode_board): [...]}。無ければDEFAULT_WINDOWS
# ボード単位のスペクトル(spec1hit_board_upic*.png)で2つの山が見える組を指定する。
# 例(run13-17、μPIC1 と μPIC2 の一部):
#   (3, 3): [(550, 1100), (2200, 3900)], (3, 4): [(500, 1050), (2100, 4000)], (3, 5): [(400, 1000), (2000, 3700)],
#   (4, 3): [(500, 1100), (2300, 4200)], (4, 4): [(500, 1100), (2200, 4100)], (4, 5): [(500, 1050), (2000, 3800)],
#   (5, 3): [(750, 1100), (2000, 3800)], (5, 4): [(450, 1000), (2000, 3600)], (5, 5): [(450, 1000), (2000, 3500)],
#   (8, 6): [(400, 900), (2200, 3600)], (8, 7): [(450, 950), (1800, 3400)],
BOARD_WINDOWS = {}
AUTO_WINDOW_NSIGMA = 2.0  # ch組の範囲 = ボードの中心 ± この倍数 × ボードのσ [dimensionless]
BOARD_SIGMA_RANGE = None  # 例: (0.5, 1.2)。ボードから範囲を決めたch組で、ガウスσを (下限,上限) × ボードのσ に制限。Noneで制限なし
# run13-17 では制限してもOKの組は増えなかった(144組中 制限なし18、(0.5,1.2)で10)
MIN_EVENTS = 200      # 組ごとの最小1-hitイベント数 [events]
FIT_REBIN = 2         # フィット時にまとめる元ビン数 [bins]（1なら5 ADU x sample/bin）
CHI2NDF_MAX = 5.0     # χ²/ndf上限 [dimensionless]
SIGNIFICANCE_MIN = 3.0  # ガウス高さ/誤差の下限 [dimensionless]

In [ ]:
def two_point_calibration(mu, mu_error, energy):
    """Two-point calibration with independent peak-position errors and fixed energies."""
    mu = np.asarray(mu, dtype=float)
    err = np.asarray(mu_error, dtype=float)
    energy = np.asarray(energy, dtype=float)
    if any(x.shape != (2,) for x in (mu, err, energy)):
        raise ValueError('中心位置・誤差・エネルギーは2要素ずつ指定してください')
    if not all(np.all(np.isfinite(x)) for x in (mu, err, energy)):
        raise ValueError('中心位置・誤差・エネルギーは有限値で指定してください')
    if mu[1] <= mu[0] or energy[1] <= energy[0] or np.any(err < 0):
        raise ValueError('中心位置とエネルギーは昇順、誤差は非負で指定してください')
    delta = mu[1] - mu[0]
    de = energy[1] - energy[0]
    ka = de / delta
    kb = energy[0] - ka * mu[0]
    # Jacobian of (Ka, Kb) with respect to (mu1, mu2).
    jacobian = de / delta**2 * np.array([[1., -1.], [-mu[1], mu[0]]])
    covariance = jacobian @ np.diag(err**2) @ jacobian.T
    ka_error, kb_error = np.sqrt(np.maximum(np.diag(covariance), 0.))
    corr = float(covariance[0,1] / (ka_error * kb_error)) if ka_error * kb_error > 0 else None
    uncertainty = dict(Ka_stat_error=float(ka_error), Kb_stat_error=float(kb_error),
                       cov_Ka_Kb=float(covariance[0,1]),
                       covariance=covariance.tolist(), covariance_order=['Ka', 'Kb'],
                       corr_Ka_Kb=corr)
    return float(ka), float(kb), uncertainty

def fit_line(h, window, name, seed_mu=None, sigma_limits=None):
    lo, hi = map(float, window)
    if not h.GetXaxis().GetXmin() < lo < hi < h.GetXaxis().GetXmax():
        raise ValueError(f'{name}: fit範囲がスペクトルの軸範囲外です')
    t = h.Clone(name); t.SetDirectory(0); t.Rebin(FIT_REBIN)
    t.GetXaxis().SetRangeUser(lo, hi)
    width = hi-lo
    # 中心の初期値: 指定があればそれ(ボードの中心)、無ければ範囲内の最大ビン
    seed = seed_mu if seed_mu is not None and lo < seed_mu < hi else t.GetBinCenter(t.GetMaximumBin())
    s_lo, s_hi = sigma_limits if sigma_limits is not None else (t.GetBinWidth(1)/2, width/2)
    s_lo, s_hi = max(s_lo, t.GetBinWidth(1)/2), min(s_hi, width/2)
    if not s_lo < s_hi: raise ValueError(f'{name}: σの制限が不正です ({s_lo}, {s_hi})')
    bg = max(0, (t.GetBinContent(t.FindBin(lo))+t.GetBinContent(t.FindBin(hi)))/2)
    fn = ROOT.TF1(name+'_fit', 'gaus(0)+pol1(3)', lo, hi)
    fn.SetParameters(max(1,t.GetMaximum()-bg), seed, min(max(width/8, s_lo*1.1), s_hi*0.9), bg, 0)
    fn.SetParLimits(0, 0, max(10,10*t.GetMaximum()))
    fn.SetParLimits(1, lo, hi); fn.SetParLimits(2, s_lo, s_hi)
    status = int(t.Fit(fn, 'QSN0R'))
    mu, err, sigma = fn.GetParameter(1), fn.GetParError(1), abs(fn.GetParameter(2))
    chi = fn.GetChisquare()/fn.GetNDF() if fn.GetNDF() > 0 else np.inf
    sig = fn.GetParameter(0)/fn.GetParError(0) if fn.GetParError(0)>0 else 0
    reasons = []
    if status != 0: reasons.append(f'fit status {status}')
    if not np.isfinite(mu) or not lo+0.02*width < mu < hi-0.02*width: reasons.append('中心が範囲端')
    if not s_lo*1.02 < sigma < s_hi*0.98: reasons.append('幅が制限端')
    if not 0 < err < abs(mu)*0.1: reasons.append('中心誤差が大きい')
    if not chi < CHI2NDF_MAX: reasons.append('chi2/ndf')
    if not sig >= SIGNIFICANCE_MIN: reasons.append('有意度不足')
    return dict(mu=mu, error=err, sigma=sigma, chi2ndf=chi, significance=sig,
                reasons=reasons, fn=fn, window=(lo,hi))

if ENERGIES_KEV is None or len(ENERGIES_KEV)!=2 or not 0 < ENERGIES_KEV[0] < ENERGIES_KEV[1]:
    raise ValueError('確認済みの2本の線をENERGIES_KEVに小さい順で設定してください')
if DEFAULT_WINDOWS is None and not WINDOWS and not BOARD_WINDOWS:
    raise ValueError('スペクトルを確認してDEFAULT_WINDOWS・BOARD_WINDOWS・WINDOWSのどれかを設定してください')
if not AUTO_WINDOW_NSIGMA > 0: raise ValueError('AUTO_WINDOW_NSIGMAは正の値にしてください')
if BOARD_SIGMA_RANGE is not None and not 0 < BOARD_SIGMA_RANGE[0] < BOARD_SIGMA_RANGE[1]:
    raise ValueError('BOARD_SIGMA_RANGEは (下限, 上限) で 0 < 下限 < 上限 にしてください')
assert isinstance(FIT_REBIN, int) and FIT_REBIN > 0 and 2000 % FIT_REBIN == 0

In [ ]:
# 1. ボード単位でピーク位置を決める
board_fits, board_ok = {}, set()
for pair, h in board_spectra.items():
    windows = BOARD_WINDOWS.get(pair, DEFAULT_WINDOWS)
    if windows is None: print(f'board A{pair[0]} C{pair[1]}: fit範囲未指定'); continue
    if h.Integral() < MIN_EVENTS: print(f'board A{pair[0]} C{pair[1]}: 統計不足'); continue
    if len(windows)!=2 or not windows[0][1] < windows[1][0]:
        raise ValueError(f'board {pair}: 低い線、高い線の順で重ならない2範囲を指定してください')
    r = [fit_line(h, w, f'fit_board_A{pair[0]}_C{pair[1]}_{k}') for k, w in enumerate(windows)]
    board_fits[pair] = r
    why = [f'line{k}: {s}' for k, v in enumerate(r) for s in v['reasons']]
    if not r[0]['mu'] < r[1]['mu']: why.append('ピークの順序が不正')
    if why: print(f'board A{pair[0]} C{pair[1]}:', ', '.join(why))
    else: board_ok.add(pair)
print('ボード単位でフィットが通った組:', len(board_ok), '/', len(board_spectra))
for (a, c) in sorted(board_ok):
    r = board_fits[a, c]
    print(f'  board A{a} C{c}: ' + ', '.join(f'{ENERGIES_KEV[k]:g} keV: {v["mu"]:.0f} ± {v["sigma"]:.0f} (σ)'
                                            for k, v in enumerate(r)))
for panel in range(4):
    plot_spectra_grid(board_spectra, range(3*panel, 3*(panel+1)), 'B',
                      f'μPIC{panel}: board 1-hit spectra with fits ({ENERGIES_KEV[0]:g} keV: orange, {ENERGIES_KEV[1]:g} keV: green)',
                      OUT/f'spec1hit_board_fit_upic{panel}.png', figsize=(9, 11), fs=9,
                      fit_results=board_fits, ok=board_ok)

# 2. 各ch組のフィット範囲を決める: WINDOWS > ボードから自動 > DEFAULT_WINDOWS
def board_window(board):
    """ボードの組 board のフィットがOKなら、ボードの中心 ± AUTO_WINDOW_NSIGMA × σ の範囲を返す。"""
    if board not in board_ok: return None
    r = board_fits[board]
    w = [[v['mu'] - AUTO_WINDOW_NSIGMA*v['sigma'], v['mu'] + AUTO_WINDOW_NSIGMA*v['sigma']] for v in r]
    if w[0][1] >= w[1][0]:  # 重なれば2つの中心の中点で分ける
        mid = (r[0]['mu'] + r[1]['mu']) / 2
        w[0][1], w[1][0] = mid, mid + 5*FIT_REBIN
    h = board_spectra[board]; xmin, xmax = h.GetXaxis().GetXmin(), h.GetXaxis().GetXmax()
    return [(max(lo, xmin + 5*FIT_REBIN), min(hi, xmax - 5*FIT_REBIN)) for lo, hi in w]

windows_used, window_source = {}, {}
for pair in spectra:
    if pair in WINDOWS: windows_used[pair], window_source[pair] = WINDOWS[pair], 'WINDOWS'
    elif board_window((pair[0]//4, pair[1]//4)) is not None:
        windows_used[pair], window_source[pair] = board_window((pair[0]//4, pair[1]//4)), 'board'
    elif DEFAULT_WINDOWS is not None: windows_used[pair], window_source[pair] = DEFAULT_WINDOWS, 'DEFAULT_WINDOWS'
from collections import Counter
print('ch組の範囲の決め方:', dict(Counter(window_source.values())), '/ 範囲なし:', len(spectra) - len(windows_used))

In [ ]:
# 3. 各ch組をフィットして係数を決める
fits, cal, cal_uncertainty, rejected = {}, {}, {}, {}
for pair, h in spectra.items():
    windows = windows_used.get(pair)
    if windows is None: rejected[pair] = ['fit範囲未指定']; continue
    if h.Integral() < MIN_EVENTS: rejected[pair] = ['統計不足']; continue
    if len(windows)!=2 or not windows[0][1] < windows[1][0]:
        raise ValueError(f'{pair}: 低い線、高い線の順で重ならない2範囲を指定してください')
    if window_source[pair] == 'board':  # ボードの中心を初期値に、σをボードのσの範囲に制限
        b = board_fits[pair[0]//4, pair[1]//4]
        opts = [dict(seed_mu=v['mu'], sigma_limits=None if BOARD_SIGMA_RANGE is None else
                     (BOARD_SIGMA_RANGE[0]*v['sigma'], BOARD_SIGMA_RANGE[1]*v['sigma'])) for v in b]
    else:
        opts = [{}, {}]
    r = [fit_line(h,w,f'fit_A{pair[0]}_C{pair[1]}_{k}', **opts[k]) for k,w in enumerate(windows)]
    fits[pair] = r
    why = [f'line{k}: {s}' for k,v in enumerate(r) for s in v['reasons']]
    if not r[0]['mu'] < r[1]['mu']: why.append('ピークの順序が不正')
    if why: rejected[pair] = why; continue
    ka, kb, uncertainty = two_point_calibration(
        [v['mu'] for v in r], [v['error'] for v in r], ENERGIES_KEV)
    if not np.isfinite(ka+kb) or ka<=0: rejected[pair] = ['係数が不正']; continue
    cal[pair] = (ka,kb)
    cal_uncertainty[pair] = uncertainty
print('較正成功:',len(cal), '/ 576組')
for pair, why in rejected.items(): print(pair, ', '.join(why))

In [ ]:
# 4. 2 ch単位のフィット(確認用。ゲインマップの係数には使わない)
# 範囲と初期値はch組と同じくボード(2 ch番号 // 2)の結果から決め、無ければDEFAULT_WINDOWS
pair2_fits, pair2_ok, pair2_rejected = {}, set(), {}
for pair, h in pair2_spectra.items():
    board = (pair[0]//2, pair[1]//2)
    windows = board_window(board) or DEFAULT_WINDOWS
    if windows is None: pair2_rejected[pair] = ['fit範囲未指定']; continue
    if h.Integral() < MIN_EVENTS: pair2_rejected[pair] = ['統計不足']; continue
    if board in board_ok:
        opts = [dict(seed_mu=v['mu'], sigma_limits=None if BOARD_SIGMA_RANGE is None else
                     (BOARD_SIGMA_RANGE[0]*v['sigma'], BOARD_SIGMA_RANGE[1]*v['sigma'])) for v in board_fits[board]]
    else:
        opts = [{}, {}]
    r = [fit_line(h, w, f'fit_2ch_A{pair[0]}_C{pair[1]}_{k}', **opts[k]) for k, w in enumerate(windows)]
    pair2_fits[pair] = r
    why = [f'line{k}: {s}' for k, v in enumerate(r) for s in v['reasons']]
    if not r[0]['mu'] < r[1]['mu']: why.append('ピークの順序が不正')
    if why: pair2_rejected[pair] = why
    else: pair2_ok.add(pair)
print('2 ch単位でフィットが通った組:', len(pair2_ok), '/', len(pair2_spectra))
for panel in range(4):
    plot_spectra_grid(pair2_spectra, range(6*panel, 6*(panel+1)), 'G',
                      f'μPIC{panel}: 2-ch 1-hit spectra with fits ({ENERGIES_KEV[0]:g} keV: orange, {ENERGIES_KEV[1]:g} keV: green)',
                      OUT/f'spec1hit_2ch_fit_upic{panel}.png', figsize=(12, 15), fs=8,
                      fit_results=pair2_fits, ok=pair2_ok)

## 5. 各組のフィットと係数マップを確認

フィットした各組のスペクトルを `qa/Axx_Cxx.png` に保存します。
自動判定に落ちた組も図で確認できます。
μPICごとに全組のフィット結果を並べた図を `spec1hit_fit_upic{0..3}.png` に保存します（3章の一覧と同じ並び・横軸）。
フィット曲線（低い線: オレンジ、高い線: 緑）と中心（点線）を重ね、較正に使える組を緑のOK、それ以外を赤のNG（背景も赤）で示します。
NGの理由は4章のセルの出力または `gain_map.json` の `rejected` を見てください。
判定を通っても、線を取り違えていないか確認してください。
ボード単位のフィット図（`spec1hit_board_fit_upic{0..3}.png`）は4章で描いています。ボードの係数はゲインマップには使いません。
係数の白い領域は未較正です。マップは物理XYではなくFADC chの組の並びです。

In [ ]:
QA = OUT / 'qa'
QA.mkdir(exist_ok=True)
for pair, rr in fits.items():
    h = spectra[pair]
    x = np.array([h.GetBinCenter(i) for i in range(1,h.GetNbinsX()+1)])
    y = np.array([h.GetBinContent(i) for i in range(1,h.GetNbinsX()+1)])
    fig, ax = plt.subplots(figsize=(8,4))
    ax.step(x,y,where='mid',lw=0.7)
    for k,r in enumerate(rr):
        xx = np.linspace(*r['window'], 200)
        # ROOT fit values are counts per rebinned bin; compare at original bin width.
        ax.plot(xx, [r['fn'].Eval(v)/FIT_REBIN for v in xx], label=f'{ENERGIES_KEV[k]:g} keV')
        ax.axvline(r['mu'], ls=':',color='k')
    ax.set(xlim=(rr[0]['window'][0],rr[1]['window'][1]), xlabel='Anode charge [ADU x sample]',
           ylabel='Events / 5 ADU x sample', title=f'A{pair[0]} C{pair[1]}: '+('OK' if pair in cal else 'NG'))
    ax.legend(); fig.tight_layout(); fig.savefig(QA/f'A{pair[0]:02d}_C{pair[1]:02d}.png',dpi=100); plt.close(fig)
fig, axs = plt.subplots(1,2,figsize=(15,6))
for k, ax in enumerate(axs):
    z = np.full((48,48),np.nan)
    for (a,c), v in cal.items(): z[c,a] = v[k]
    m = ax.imshow(np.ma.masked_invalid(z),origin='lower')
    fig.colorbar(m,ax=ax,label=['Ka [keV/(ADU x sample)]','Kb [keV]'][k])
    ax.set(xlabel='Anode FADC ch',ylabel='Cathode FADC ch',title=['Ka','Kb'][k])
fig.tight_layout(); fig.savefig(OUT/'gain_map.png',dpi=150); plt.show()
if fits:
    pair = next(iter(fits)); display(Image(filename=str(QA/f'A{pair[0]:02d}_C{pair[1]:02d}.png')))

fig, axs = plt.subplots(1, 2, figsize=(15, 6))
for ax, field, label in zip(axs, ['Ka_stat_error', 'Kb_stat_error'],
                           ['Ka statistical error [keV/(ADU x sample)]', 'Kb statistical error [keV]']):
    z = np.full((48, 48), np.nan)
    for (a, c), uncertainty in cal_uncertainty.items(): z[c, a] = uncertainty[field]
    m = ax.imshow(np.ma.masked_invalid(z), origin='lower')
    fig.colorbar(m, ax=ax, label=label)
    ax.set(xlabel='Anode FADC ch', ylabel='Cathode FADC ch', title=field)
fig.tight_layout(); fig.savefig(OUT/'gain_map_errors.png', dpi=150); plt.show()
for pair in sorted(cal)[:5]:
    ka, kb = cal[pair]; u = cal_uncertainty[pair]
    print(f'A{pair[0]} C{pair[1]}: Ka={ka:.6g} +/- {u["Ka_stat_error"]:.2g}, '
          f'Kb={kb:.6g} +/- {u["Kb_stat_error"]:.2g}, corr={u["corr_Ka_Kb"]:.4f}')


In [ ]:
# μPICごとのフィット結果一覧(フィットした組だけでなく、統計不足・範囲未指定の組もNGとして表示)
for panel in range(4):
    plot_spectra_grid(spectra, range(12*panel, 12*(panel+1)), '',
                      f'μPIC{panel}: 1-hit spectra with fits ({ENERGIES_KEV[0]:g} keV: orange, {ENERGIES_KEV[1]:g} keV: green)',
                      OUT/f'spec1hit_fit_upic{panel}.png', figsize=(16, 20), fit_results=fits, ok=set(cal))

## 6. ゲインマップを保存する

次のセルは、較正結果を冒頭の `OUT` に書き出します。既存の較正ファイルは書き換えません。

| ファイル | 中身 |
|---|---|
| `gain_map.dat` | 解析チェーンの `TPC_Calibration` が読む係数の表。1行が1組で `anode_ch cathode_ch Ka Kb` の4列。同じμPIC内の組（12 × 12組 × 4 μPIC = 576組）を全部書く |
| `gain_map.json` | 作り方の記録。入力ファイル、設定、各組のフィット結果、NGの理由、係数の誤差 |

`TPC_Calibration` の読み込みはコメント行や見出しに対応していません。そのため `gain_map.dat` には数値だけを書き、それ以外の情報は `gain_map.json` に分けています。

### 較正できなかった組
- `gain_map.dat` では Ka = Kb = 0 にします。周りの組の値などで埋めることはしません。
- `TPC_Calibration` はアノードchごとに「電荷 × Ka + Kb」を足してエネルギーを出します。そのため、較正できなかった組に入った電荷は 0 keV として扱われ、そのイベントのエネルギーは低めに出ます。
- 解析する前に、使う領域が較正済みかどうかを5章の `gain_map.png` で確認してください（白い所が未較正）。

### 係数の誤差
- `gain_map.json` の `calibration_uncertainty` に、組ごとの Ka・Kb の標準誤差、共分散（順序は Ka, Kb）、相関係数を書きます。誤差のマップは5章の `gain_map_errors.png` です。
- これは2本のピーク中心の誤差だけから計算した統計誤差です（計算式は4章）。線の同定やフィット範囲による系統誤差は含みません。
- 較正できなかった組には誤差を書きません（0 とはしません）。
- 解析チェーンは誤差を使いません（`gain_map.dat` は4列のまま）。エネルギーの誤差が必要なときは、4章の式で自分で計算してください。

### SMILE-2+ 版との違い
SMILE-2+ 版の `TPC_Calibration_2p` には、エネルギーを足す前に組を選ぶ条件が2つあります。

- ASICの端から4本以内のストリップを除く
- 同時計数が100を超えるパネルの組だけを足す

SMILE-3 で使う `TPC_Calibration` には、この2つの条件がありません。このノートブックでも入れていません。

In [ ]:
if not cal: raise RuntimeError('較正できた組がないためゲインマップを書きません')
CAL = OUT / 'gain_map.dat'
with CAL.open('w') as stream:
    for c in range(48):
        for a in range(12*(c//12),12*(c//12+1)):
            ka,kb = cal.get((a,c),(0.0,0.0))
            stream.write(f'{a} {c} {ka:.12g} {kb:.12g}\n')
report = {'inputs':[str(p.resolve()) for p in RAWS], 'energies_keV':ENERGIES_KEV,
          'charge_unit':'ADU x sample', 'Ka_unit':'keV/(ADU x sample)', 'Kb_unit':'keV',
          'base_window_samples':BASE_WINDOW, 'threshold_ADU':THRESHOLD,
          'default_windows_charge':DEFAULT_WINDOWS,
          'windows_charge':{f'{a},{c}':v for (a,c),v in WINDOWS.items()},
          'board_windows_charge':{f'{a},{c}':v for (a,c),v in BOARD_WINDOWS.items()},
          'auto_window_nsigma':AUTO_WINDOW_NSIGMA, 'board_sigma_range':BOARD_SIGMA_RANGE,
          'board_fits':{f'{a},{c}':[{k:v for k,v in r.items() if k!='fn'} for r in rr] for (a,c),rr in board_fits.items()},
          'board_ok':[f'{a},{c}' for (a,c) in sorted(board_ok)],
          'window_source':{f'{a},{c}':v for (a,c),v in window_source.items()},
          'pair2_fits':{f'{a},{c}':[{k:v for k,v in r.items() if k!='fn'} for r in rr] for (a,c),rr in pair2_fits.items()},
          'pair2_ok':[f'{a},{c}' for (a,c) in sorted(pair2_ok)],
          'fit_rebin':FIT_REBIN, 'min_events':MIN_EVENTS, 'chi2ndf_max':CHI2NDF_MAX,
          'significance_min':SIGNIFICANCE_MIN,
          'uncertainty_assumptions': {'peak_centers_independent': True, 'energies_fixed': True,
                                      'method': 'first-order Jacobian propagation', 'systematics_included': False},
          'uncertainty_units': {'Ka_stat_error': 'keV/(ADU x sample)', 'Kb_stat_error': 'keV',
                                'cov_Ka_Kb': 'keV^2/(ADU x sample)'},
          'calibration_uncertainty': {f'{a},{c}': u for (a,c),u in cal_uncertainty.items()},
          'calibrated':{f'{a},{c}':list(v) for (a,c),v in cal.items()},
          'rejected':{f'{a},{c}':v for (a,c),v in rejected.items()},
          'fits':{f'{a},{c}':[{k:v for k,v in r.items() if k!='fn'} for r in rr] for (a,c),rr in fits.items()}}
(OUT/'gain_map.json').write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n')
print('wrote',CAL, f'({len(cal)}/576 calibrated)')

### 解析で使う
`gain_map.dat` を確認したら、解析を実行するときに次の2つを指定します。

```
--set TPC_Calibration.NoCalData=false
--set TPC_Calibration.ParameterFile=<gain_map.datの絶対パス>
```

解析時の波形の積分条件（ベースラインの長さ `BASE_WINDOW`、積分の閾値 `THRESHOLD`）は、較正に使った値と同じにしてください。違うと電荷の値が変わり、係数が合わなくなります。